# Solutions Notebook

## Module 5: Automation in AI | Al Jazira Bank

This notebook contains reference solutions for the data analysis exercises in Day 1 and Day 2 notebooks. Use this for facilitator review and post-session reference only.

### Contents
1. Day 1: Workflow analysis solutions
2. Day 2: Automation design solutions
3. Expected outputs and interpretation notes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

workflows = pd.read_csv("../data/workflow_inventory.csv")
candidates = pd.read_csv("../data/automation_candidates.csv")

print("Datasets loaded successfully.")
print(f"Workflows: {len(workflows)} rows")
print(f"Candidates: {len(candidates)} rows")

In [ ]:
# Solution: Day 1 - Top candidates by composite score
#
# Expected top 5 using equal-weighted hours + error rate:
# 1. W001 Payment reconciliation (42 hrs, 3.8% error)
# 2. W002 Customer onboarding document processing (28 hrs, 6.2% error)
# 3. W006 KYC document review (35 hrs, 4.3% error) - note: Medium potential, High complexity
# 4. W005 Credit card dispute intake (16 hrs, 7.5% error)
# 5. W010 Incident ticket triage (24 hrs, 5.1% error)
#
# Key teaching points:
# - W006 (KYC) scores high but has High complexity. Strong answers discuss this trade-off.
# - W005 (disputes) has the highest error rate but moderate hours. It may be a better
#   quality-improvement target than a cost-saving target.
# - Workflows with Low automation potential (W007, W013, W017, W020) should be excluded
#   or discussed as "not yet ready" candidates.

analysis = workflows.copy()
analysis["hours_norm"] = analysis["manual_hours_weekly"] / analysis["manual_hours_weekly"].max()
analysis["error_norm"] = analysis["error_rate_pct"] / analysis["error_rate_pct"].max()
analysis["composite"] = analysis["hours_norm"] * 0.5 + analysis["error_norm"] * 0.5

high_med = analysis[analysis["automation_potential"].isin(["High", "Medium"])]
top5 = high_med.sort_values("composite", ascending=False).head(5)

print("Day 1 Solution: Top 5 candidates by composite score")
print(top5[["workflow_id", "process_name", "department",
            "manual_hours_weekly", "error_rate_pct",
            "automation_potential", "complexity", "composite"]].to_string(index=False))

In [ ]:
# Solution: Day 2 - Prioritisation matrix and ROI proxy
#
# Expected Quick Wins (high savings, low effort):
# - W001 Payment reconciliation (30 hrs saved, Medium effort)
# - W004 Internal transfer processing (18 hrs saved, Low effort)
# - W003 Daily transaction reporting (14 hrs saved, Low effort)
#
# Expected Strategic Bets (high savings, high effort):
# - W006 KYC document review (20 hrs saved, High effort)
# - W002 Customer onboarding (18 hrs saved, Medium effort)
#
# Key teaching points:
# - ROI proxy (savings / effort) favours low-effort, high-savings candidates.
# - W004 and W003 are strong first projects because they are rule-based, low risk,
#   and do not require human review.
# - W006 and W011 are valuable but complex. They should be second-wave candidates
#   after simpler automations prove the approach.

combined = workflows.merge(candidates, on="workflow_id", how="inner")
effort_map = {"Low": 1, "Medium": 2, "High": 3}
combined["effort_num"] = combined["implementation_effort"].map(effort_map)
combined["roi_proxy"] = combined["estimated_saving_hours"] / combined["effort_num"]

final = combined.sort_values("roi_proxy", ascending=False)

print("Day 2 Solution: All candidates ranked by ROI proxy")
print(final[["process_name", "automation_type", "estimated_saving_hours",
             "implementation_effort", "risk_level", "requires_human_review",
             "roi_proxy"]].to_string(index=False))

print("\nRecommended implementation sequence:")
print("Phase 1 (Quick wins): Internal transfer processing, Daily transaction reporting,")
print("  Payment reconciliation")
print("Phase 2 (Medium complexity): Account closure, Access rights provisioning,")
print("  Incident ticket triage")
print("Phase 3 (Strategic): KYC document review, Customer onboarding,")
print("  Personal loan processing")

In [ ]:
# Solution: Summary visualisation
#
# Combined view showing savings potential vs risk for all candidates.

risk_map = {"Low": 1, "Medium": 2, "High": 3}
combined["risk_num"] = combined["risk_level"].map(risk_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Savings by automation type
type_savings = combined.groupby("automation_type")["estimated_saving_hours"].sum()
type_savings.plot(kind="bar", ax=axes[0], color=["#3498db", "#2ecc71"])
axes[0].set_title("Total Estimated Savings by Automation Type")
axes[0].set_ylabel("Hours per Week")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

# Chart 2: Savings vs Risk
colours_risk = {1: "#2ecc71", 2: "#f39c12", 3: "#e74c3c"}
for risk_val, group in combined.groupby("risk_num"):
    label = {1: "Low", 2: "Medium", 3: "High"}[risk_val]
    axes[1].scatter(
        group["risk_num"],
        group["estimated_saving_hours"],
        label=label,
        c=colours_risk[risk_val],
        s=100,
        alpha=0.8,
    )
    for _, row in group.iterrows():
        axes[1].annotate(
            row["process_name"][:18],
            (row["risk_num"], row["estimated_saving_hours"]),
            fontsize=7,
            ha="left",
        )

axes[1].set_title("Estimated Savings vs Risk Level")
axes[1].set_xlabel("Risk Level (1=Low, 2=Medium, 3=High)")
axes[1].set_ylabel("Estimated Savings (hours/week)")
axes[1].legend(title="Risk")

plt.tight_layout()
plt.show()

print("\nKey insight: Rule-based automations deliver the most total savings")
print("with lower risk. AI-assisted automations target higher-value but")
print("more complex processes that require human review safeguards.")